# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s-nancyzakria-hash/flyrank-intern-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Logistic Regression as a interpretable baseline,
# followed by Random Forest Classifier. Random Forest is selected because it effectively handles non-linear relationships and missing values without requiring complex scaling.
# Additionally, it avoids overfitting through ensemble tree aggregation.
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit


df = pd.read_csv('/content/content_refresh_anonymized.csv')

print(f'Dataset shape: {df.shape}')

# Create binary target from trend_direction if is_declining_label doesn't exist
if 'is_declining_label' not in df.columns:
  df['is_declining_label'] = (
      df['trend_direction'].astype(str).str.lower() == 'down'
  ).astype(int)

print(
    'Base Rate (Target Distribution):\n'
    f'{df["is_declining_label"].value_counts(normalize=True)}'
)
print(f'Dataset shape: {df.shape}')
print(f'Base Rate (Target Distribution):\n{df["trend_direction"].value_counts(normalize=True)}')

Dataset shape: (30000, 44)
Base Rate (Target Distribution):
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64
Dataset shape: (30000, 45)
Base Rate (Target Distribution):
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

#Grouped Train/Validation Split by client_id (80% train, 20% validation). Content items belonging to the same client share common publishing behaviors
# Splitting randomly across rows would cause data leakage between train and validation sets. Grouping by client_id ensures an evaluation on unseen clients.
# Handle Data Leakage: Exclude target derivatives and pseudonyms

# Prevent Data Leakage & Drop Pseudonyms
LEAKAGE_COLS = ['trend_direction', 'trend_pct']  # Source of label!
ID_COLS = ['content_id', 'client_id']
TARGET_COL = 'is_declining_label'

X_raw = df.drop(
    columns=LEAKAGE_COLS + ID_COLS + [TARGET_COL], errors='ignore'
).copy()

# Handle missing values
for col in X_raw.columns:
  if X_raw[col].isnull().sum() > 0:
    X_raw[f'has_{col}'] = X_raw[col].notnull().astype(int)

# Handle avg_position == 0 (0 means "no data", not rank zero)
if 'avg_position' in X_raw.columns:
  X_raw['avg_position_missing'] = (X_raw['avg_position'] == 0).astype(int)

# One-Hot Encoding for categorical/text columns
X_encoded = pd.get_dummies(X_raw, drop_first=True)

# Fill remaining missing numerical values with Median
X_clean = X_encoded.fillna(X_encoded.median(numeric_only=True))

y = df[TARGET_COL].values
groups = df['client_id'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X_clean, y, groups=groups))

X_train, X_val = X_clean.iloc[train_idx], X_clean.iloc[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(
    f'Train set: {X_train.shape[0]} rows | Validation set:'
    f' {X_val.shape[0]} rows'
)
print(
    f'Train clients: {len(np.unique(groups[train_idx]))} | Val clients:'
    f' {len(np.unique(groups[val_idx]))}'
)

Train set: 23837 rows | Validation set: 6163 rows
Train clients: 25 | Val clients: 7


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Majority Class Baseline
majority_class = pd.Series(y_train).mode()[0]
y_pred_baseline = np.full_like(y_val, fill_value=majority_class)

# 2. Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_val)
y_prob_lr = log_reg.predict_proba(X_val)[:, 1]

# 3. Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=6, random_state=42
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_val)
y_prob_rf = rf_model.predict_proba(X_val)[:, 1]

# Comparison Table
results = [
    {
        'Model': 'Baseline (Majority Class)',
        'Accuracy': accuracy_score(y_val, y_pred_baseline),
        'Precision': precision_score(y_val, y_pred_baseline, zero_division=0),
        'Recall': recall_score(y_val, y_pred_baseline, zero_division=0),
        'F1-Score': f1_score(y_val, y_pred_baseline, zero_division=0),
        'ROC-AUC': 0.5,
    },
    {
        'Model': 'Logistic Regression',
        'Accuracy': accuracy_score(y_val, y_pred_lr),
        'Precision': precision_score(y_val, y_pred_lr, zero_division=0),
        'Recall': recall_score(y_val, y_pred_lr, zero_division=0),
        'F1-Score': f1_score(y_val, y_pred_lr, zero_division=0),
        'ROC-AUC': roc_auc_score(y_val, y_prob_lr),
    },
    {
        'Model': 'Random Forest Classifier',
        'Accuracy': accuracy_score(y_val, y_pred_rf),
        'Precision': precision_score(y_val, y_pred_rf, zero_division=0),
        'Recall': recall_score(y_val, y_pred_rf, zero_division=0),
        'F1-Score': f1_score(y_val, y_pred_rf, zero_division=0),
        'ROC-AUC': roc_auc_score(y_val, y_prob_rf),
    },
]

comparison_df = pd.DataFrame(results)
display(comparison_df)

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Baseline (Majority Class),0.510952,0.510952,1.000000,0.676332,0.500000
1,Logistic Regression,0.997566,0.998727,0.996507,0.997616,0.999984
2,Random Forest Classifier,0.661366,0.640476,0.768815,0.698802,0.721025


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [35]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Permutation Feature Importance
perm_imp = permutation_importance(
    rf_model, X_val, y_val, n_repeats=10, random_state=42
)
importance_df = pd.DataFrame({
    'Feature': X_val.columns,
    'Importance_Mean': perm_imp.importances_mean,
}).sort_values('Importance_Mean', ascending=False)

print('--- Top 5 Most Important Features ---')
display(importance_df.head(5))

# 2. Inspect 3 Concrete Misclassified Cases
val_analysis = X_val.copy()
val_analysis['Actual'] = y_val
val_analysis['Predicted'] = y_pred_rf
val_analysis['Prob_Declining'] = y_prob_rf

wrong_cases = val_analysis[
    val_analysis['Actual'] != val_analysis['Predicted']
].head(3)

top_3_features = list(importance_df.head(3)['Feature'])
print('\n--- 3 Misclassified Cases ---')
display(
    wrong_cases[['Actual', 'Predicted', 'Prob_Declining'] + top_3_features]
)

--- Top 5 Most Important Features ---


,Feature,Importance_Mean
18,impressions_prev_30d,0.113549
15,impressions_last_30d,0.047185
16,clicks_last_30d,0.006004
67,impression_tier_low,0.005355
17,sessions_last_30d,0.003618



--- 3 Misclassified Cases ---


,Actual,Predicted,Prob_Declining,impressions_prev_30d,impressions_last_30d,clicks_last_30d
1,1,0,0.446520,5915,2501,2
13,0,1,0.758532,77,85,0
26,0,1,0.714805,818,743,0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.